In [1]:
#all necessary imports
import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import os

# TA indicators
import ta
from ta.trend import PSARIndicator

# Data Normalization and Scaling
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Scikit-learn utilities
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# hyperparameter optimization
import optuna
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", device)

print("Torch version:", torch.__version__)
print("Polars version:", pl.__version__)
print("TQDM imported successfully!")
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("TA library imported successfully!")
print("Optuna version:",optuna.__version__)

Training device: cuda
Torch version: 2.5.1+cu121
Polars version: 0.20.26
TQDM imported successfully!
NumPy version: 1.26.4
Pandas version: 2.2.2
TA library imported successfully!
Optuna version: 4.8.0


c:\Users\User\miniconda3\envs\quant_dl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#special case company
"""
"4348"
'2285', '4081', '2286', '4340', '4342', '2283', 
'2284', '1323', '4009', '4334', '4084', '4261', 
'4012', '1830', '4333', '1835', '4165', '4347', 
'2382', '2223', '4163', '4262', '4083', '4164', 
'1834', '1303', '2081', '4162', '7204', '7202', 
'2281', '1833', '4143', '4142', '4263', '1831', 
'4051', '1322', '1182', '6017', '4161', '4031', 
'4071', '4322', '4264', '4016', '4072', '4191', 
'4332', '2381', '6014', '4017', '4015', '4324', 
'6015', '4336', '4338', '4082', '4144', '4014', 
'1111', '4291', '4339', '4013', '2282', '4325', 
'1304', '7203', '4018', '2084', '7200', '2287', 
'4331', '8313', '4193', '1183', '4192', '4345', 
'4344', '6016', '2082', '2083', '6013', '3007',
# collapse problem
"8200", "1050","1030", "1150", "4003", "2020", "1140","7020"


"""

# done training
"""
"8300","1010", "8100","2050", "2060", "1180", "3080", "3090","2370", 
"""



symbols = [

    

    # "3030", "1020",
    # "2150", "4260", "3020", "4001", "1320", "4007", "3040",
    # "8280", "6070", "3010", "2270", "3003", "1120", "8170",
    # "1214", "8010", "3002", "3050", "8030", "4250", "3005",
    # "2230", "4280", "4020", "4004", "8210", "2280", "1212",
    # "4002", "8012", "4150", "8160", "8070", "3004", "2080",
    # "3060", "1302", "8020", "4300", "4090", "6001", "6004",
    # "7040", "2310", "4006", "2300", "8230", "4200", "4100",
    # "2070", "2190", "2250", "1211", "8180", "2290", "2240",
    # "6020", "1301", "4180", "8240", "4210", "8040", "4130",
    # "8310", "4290", "4050", "4040", "1210", "4080", "2010",
    # "4170", "4240"
    ]



dfs = {}

for s in symbols:
    filepath = f"{s}_daily.csv"
    
    if not os.path.exists(filepath):
        print(f"  WARNING — {filepath} not found, skipping.")
        continue
    
    df = pd.read_csv(filepath)
    dfs[s] = df
    print(f"\n── {s} ──────────────────────────")
    print(f"  Shape : {df.shape}")
    print(f"  Head  :\n{df.head()}")
    print(f"  Dtypes:\n{df.dtypes}")
    print(f"  Nulls :\n{df.isnull().sum()}")

print(f"\n── Loaded {len(dfs)}/{len(symbols)} tickers ──")


── 7020 ──────────────────────────
  Shape : (5253, 46)
  Head  :
              datetime  ticker  open_price  high_price  low_price  \
0  2005-02-12 16:00:00    7020   55.801119   55.980544  55.406385   
1  2005-02-13 16:00:00    7020   55.549925   56.267624  55.119305   
2  2005-02-14 16:00:00    7020   55.729349   56.447049  55.514040   
3  2005-02-15 16:00:00    7020   56.124084   57.057093  55.872889   
4  2005-02-16 16:00:00    7020   56.267624   56.698243  55.908774   

   close_price        volume     EMA_10     EMA_20  EMA_ratio  ...  \
0    55.621695  3.054079e+06  54.994023  54.062946   1.028832  ...   
1    55.801119  5.494488e+06  55.140768  54.228486   1.029000  ...   
2    56.052314  4.738398e+06  55.306504  54.402184   1.030332  ...   
3    56.411164  7.855644e+06  55.507351  54.593515   1.033294  ...   
4    56.195854  4.907181e+06  55.632533  54.746119   1.026481  ...   

   volume_lag_3  volume_lag_4  volume_lag_5  above_ema10  above_ema20  \
0  3.219412e+06  5.15943

In [3]:
for symbol in symbols:
    df = pd.read_csv(f'{symbol}_daily.csv', parse_dates=["datetime"])
    
    # add your lag features + volume_pct_change_log here same as your pipeline
    for lag in range(1, 6):
        df[f"close_lag_{lag}"]   = df["close_price"].shift(lag)
        df[f"returns_lag_{lag}"] = df["close_price"].pct_change(1).shift(lag) * 100
        df[f"volume_lag_{lag}"]  = df["volume"].shift(lag)

    # Regime and momentum persistence features
    df["above_ema10"] = (df["close_price"] > df["EMA_10"]).astype(float)
    df["above_ema20"] = (df["close_price"] > df["EMA_20"]).astype(float)
    df["roc5_pos"]    = (df["ROC_5"] > 0).astype(float)
    df["roc20_pos"]   = (df["ROC_20"] > 0).astype(float)
    daily_up = df["close_price"].diff(1).gt(0).astype(int)
    streak_id = (daily_up != daily_up.shift()).cumsum()
    df["up_streak"] = (daily_up.groupby(streak_id).cumcount() + 1) * daily_up
    df["volume_log"]            = np.log1p(df["volume"])
    df["volume_pct_change_log"] = df["volume_log"].diff(1).shift(-1)
    df = df.dropna().reset_index(drop=True)

    # outlier removal
    for col, sigma in {'price_pct_change': 3, 'volume_pct_change_log': 2}.items():
        mean, std = df[col].mean(), df[col].std()
        df = df[df[col].between(mean - sigma * std, mean + sigma * std)]
    df = df.reset_index(drop=True)

    # split
    n = len(df)
    train_df = df.iloc[:int(n * 0.70)]
    val_df   = df.iloc[int(n * 0.70):int(n * 0.85)]

    # scale
    target_scaler = StandardScaler()
    train_y = target_scaler.fit_transform(train_df[['price_pct_change', 'volume_pct_change_log']])
    val_y   = target_scaler.transform(val_df[['price_pct_change', 'volume_pct_change_log']])

    print(f"\n{symbol}")
    print(f"  Raw train UP%      : {(train_df['price_pct_change'] > 0).mean():.2%}")
    print(f"  Raw val UP%        : {(val_df['price_pct_change'] > 0).mean():.2%}")
    print(f"  Scaled train mean  : {train_y[:, 0].mean():.4f}")  # should be ~0
    print(f"  Scaled train std   : {train_y[:, 0].std():.4f}")   # should be ~1
    print(f"  Scaled val mean    : {val_y[:, 0].mean():.4f}")    # if far from 0, val is different regime
    print(f"  Scaled val std     : {val_y[:, 0].std():.4f}")
    print(f"  Train date range   : {train_df['datetime'].min().date()} → {train_df['datetime'].max().date()}")
    print(f"  Val date range     : {val_df['datetime'].min().date()} → {val_df['datetime'].max().date()}")


7020
  Raw train UP%      : 44.48%
  Raw val UP%        : 46.88%
  Scaled train mean  : 0.0000
  Scaled train std   : 1.0000
  Scaled val mean    : -0.0158
  Scaled val std     : 0.8469
  Train date range   : 2005-02-20 → 2020-01-21
  Val date range     : 2020-01-22 → 2023-03-08

2370
  Raw train UP%      : 43.67%
  Raw val UP%        : 43.51%
  Scaled train mean  : -0.0000
  Scaled train std   : 1.0000
  Scaled val mean    : -0.0255
  Scaled val std     : 1.0120
  Train date range   : 2008-02-05 → 2021-01-03
  Val date range     : 2021-01-04 → 2023-09-10


In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import pickle

SEQ_LEN = 30

feature_cols = [
    # OHLCV (5)
    'open_price', 'high_price', 'low_price', 'close_price', 'volume',
    # Trend (4)
    'EMA_10', 'EMA_20', 'EMA_ratio', 'MACD_hist',
    # Momentum (4)
    'RSI_14', 'ROC_5', 'ROC_10', 'ROC_20',
    # Volatility (4)
    'ATR_14', 'ATR_ratio', 'BB_pct', 'realized_vol',
    # Volume (5)
    'OBV', 'OBV_momentum', 'volume_ratio', 'volume_surge', 'MFI_14',
    # Lag features (15)
    'close_lag_1', 'close_lag_2', 'close_lag_3', 'close_lag_4', 'close_lag_5',
    'returns_lag_1', 'returns_lag_2', 'returns_lag_3', 'returns_lag_4', 'returns_lag_5',
    'volume_lag_1', 'volume_lag_2', 'volume_lag_3', 'volume_lag_4', 'volume_lag_5',
    # Regime / persistence features (5)
    'above_ema10', 'above_ema20', 'roc5_pos', 'roc20_pos', 'up_streak',
]

target_cols = ['price_pct_change', 'volume_pct_change_log']

print(f"Total features: {len(feature_cols)}")  # should be 42


def create_sequences(X, y, seq_len):
    sequences, targets = [], []
    for i in range(seq_len, len(X)):
        sequences.append(X[i - seq_len:i])
        targets.append(y[i])
    return np.array(sequences), np.array(targets)


prepared = {}  # stores all data per symbol

for symbol in symbols:
    print(f"\n{'─'*50}")
    print(f"Processing {symbol}...")

    # ── 1. Load ───────────────────────────────────────────────────────────────
    try:
        df = pd.read_csv(f'{symbol}_daily.csv', parse_dates=["datetime"])
    except FileNotFoundError:
        print(f"  WARNING — {symbol}_daily.csv not found, skipping.")
        continue

    df = df.sort_values("datetime").reset_index(drop=True)

    # ── 2. Add lag features ───────────────────────────────────────────────────
    for lag in range(1, 6):
        df[f"close_lag_{lag}"]   = df["close_price"].shift(lag)
        df[f"returns_lag_{lag}"] = df["close_price"].pct_change(1).shift(lag) * 100
        df[f"volume_lag_{lag}"]  = df["volume"].shift(lag)

    # Regime and momentum persistence features
    df["above_ema10"] = (df["close_price"] > df["EMA_10"]).astype(float)
    df["above_ema20"] = (df["close_price"] > df["EMA_20"]).astype(float)
    df["roc5_pos"]    = (df["ROC_5"] > 0).astype(float)
    df["roc20_pos"]   = (df["ROC_20"] > 0).astype(float)
    daily_up = df["close_price"].diff(1).gt(0).astype(int)
    streak_id = (daily_up != daily_up.shift()).cumsum()
    df["up_streak"] = (daily_up.groupby(streak_id).cumcount() + 1) * daily_up

    # ── 3. Fix volume target — log return ─────────────────────────────────────
    df["volume_log"]            = np.log1p(df["volume"])
    df["volume_pct_change_log"] = df["volume_log"].diff(1).shift(-1)
    df = df.drop(columns=["volume_log"], errors="ignore")

    df = df.dropna().reset_index(drop=True)
    print(f"  Rows after lag + dropna: {len(df):,}")

    # ── 4. Outlier removal ────────────────────────────────────────────────────
    outlier_sigma = {'price_pct_change': 3, 'volume_pct_change_log': 2}
    for col, sigma in outlier_sigma.items():
        if col not in df.columns:
            continue
        mean, std = df[col].mean(), df[col].std()
        df = df[df[col].between(mean - sigma * std, mean + sigma * std)]

    df = df.reset_index(drop=True)
    print(f"  Rows after outlier removal: {len(df):,}")

    # ── 5. Percent-based split (70 / 15 / 15) ────────────────────────────────
    n         = len(df)
    train_end = int(n * 0.70)
    val_end   = int(n * 0.85)

    train_df = df.iloc[:train_end]
    val_df   = df.iloc[train_end:val_end]
    test_df  = df.iloc[val_end:]

    print(f"  Train : {len(train_df):,} ({train_df['datetime'].min().date()} → {train_df['datetime'].max().date()})")
    print(f"  Val   : {len(val_df):,} ({val_df['datetime'].min().date()} → {val_df['datetime'].max().date()})")
    print(f"  Test  : {len(test_df):,} ({test_df['datetime'].min().date()} → {test_df['datetime'].max().date()})")

    # ── 6. Check all feature cols exist ──────────────────────────────────────
    missing = [c for c in feature_cols + target_cols if c not in df.columns]
    if missing:
        print(f"  WARNING — missing columns: {missing}, skipping.")
        continue

    # ── 7. Scale ──────────────────────────────────────────────────────────────
    feature_scaler = StandardScaler()
    target_scaler  = StandardScaler()

    train_X = feature_scaler.fit_transform(train_df[feature_cols])
    val_X   = feature_scaler.transform(val_df[feature_cols])
    test_X  = feature_scaler.transform(test_df[feature_cols])

    train_y = target_scaler.fit_transform(train_df[target_cols])
    val_y   = target_scaler.transform(val_df[target_cols])
    test_y  = target_scaler.transform(test_df[target_cols])

    # ── 8. Sliding window sequences ───────────────────────────────────────────
    X_train, y_train = create_sequences(train_X, train_y, SEQ_LEN)
    X_val,   y_val   = create_sequences(val_X,   val_y,   SEQ_LEN)
    X_test,  y_test  = create_sequences(test_X,  test_y,  SEQ_LEN)

    print(f"  X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")

    # ── 9. Save scalers ───────────────────────────────────────────────────────
    os.makedirs(f'models/{symbol}', exist_ok=True)

    with open(f'models/{symbol}/{symbol}_feature_scaler.pkl', "wb") as f:
        pickle.dump(feature_scaler, f)
    with open(f'models/{symbol}/{symbol}_target_scaler.pkl', "wb") as f:
        pickle.dump(target_scaler, f)
    with open(f'models/{symbol}/{symbol}_feature_scaler.pkl', "wb") as f:
        pickle.dump(feature_scaler, f)
    with open(f'models/{symbol}/{symbol}_target_scaler.pkl', "wb") as f:
        pickle.dump(target_scaler, f)

    # ── 10. Store for model training ──────────────────────────────────────────
    prepared[symbol] = {
    "X_train": X_train, "y_train": y_train,
    "X_val":   X_val,   "y_val":   y_val,
    "X_test":  X_test,  "y_test":  y_test,
    # raw scaled — for Optuna to re-sequence with different seq_len
    "train_X_scaled": train_X,  "train_y_scaled": train_y,
    "val_X_scaled":   val_X,    "val_y_scaled":   val_y,
    "test_X_scaled":  test_X,   "test_y_scaled":  test_y,   # ← add this line
    }

print(f"\n── Done: {len(prepared)}/{len(symbol)} tickers prepared ──")

Total features: 42

──────────────────────────────────────────────────
Processing 7020...
  Rows after lag + dropna: 5,246
  Rows after outlier removal: 4,908
  Train : 3,435 (2005-02-20 → 2020-01-21)
  Val   : 736 (2020-01-22 → 2023-03-08)
  Test  : 737 (2023-03-09 → 2026-05-12)
  X_train: (3405, 30, 42) | X_val: (706, 30, 42) | X_test: (707, 30, 42)

──────────────────────────────────────────────────
Processing 2370...
  Rows after lag + dropna: 4,548
  Rows after outlier removal: 4,211
  Train : 2,947 (2008-02-05 → 2021-01-03)
  Val   : 632 (2021-01-04 → 2023-09-10)
  Test  : 632 (2023-09-11 → 2026-05-11)
  X_train: (2917, 30, 42) | X_val: (602, 30, 42) | X_test: (602, 30, 42)

── Done: 2/4 tickers prepared ──


In [5]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np


# ── Dataset ───────────────────────────────────────────────────────────────────
class TadawulDataset(Dataset):
    def __init__(self, X, y, weights=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.w = torch.tensor(weights, dtype=torch.float32) \
                 if weights is not None \
                 else torch.ones(len(X), dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.w[idx]


BATCH_SIZE = 32
loaders    = {}  # loaders["4030"] → {train, val, test}

for symbol, data in prepared.items():
    train_loader = DataLoader(TadawulDataset(data["X_train"], data["y_train"]), batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(TadawulDataset(data["X_val"],   data["y_val"]),   batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(TadawulDataset(data["X_test"],  data["y_test"]),  batch_size=BATCH_SIZE, shuffle=False)

    loaders[symbol] = {
        "train": train_loader,
        "val":   val_loader,
        "test":  test_loader,
    }

    print(f"\n{symbol} — Train: {len(train_loader)} batches | Val: {len(val_loader)} batches | Test: {len(test_loader)} batches")

    # ── Sanity check ──────────────────────────────────────────────────────────
    sample_X, sample_y, sample_w = next(iter(train_loader))   # ← unpack 3
    print(f"  Sample X: {sample_X.shape} → (batch, seq_len, features)")
    print(f"  Sample y: {sample_y.shape} → (batch, 2)")
    print(f"  Sample w: {sample_w.shape} → (batch,)  weights all 1.0 since no weights passed")


7020 — Train: 107 batches | Val: 23 batches | Test: 23 batches
  Sample X: torch.Size([32, 30, 42]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 2]) → (batch, 2)
  Sample w: torch.Size([32]) → (batch,)  weights all 1.0 since no weights passed

2370 — Train: 92 batches | Val: 19 batches | Test: 19 batches
  Sample X: torch.Size([32, 30, 42]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 2]) → (batch, 2)
  Sample w: torch.Size([32]) → (batch,)  weights all 1.0 since no weights passed


In [6]:
import optuna
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import balanced_accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

best_params_all = {}


# ── BiLSTM + Attention Model (with LayerNorm) ─────────────────────────────────
class StockPctChangeBiLSTMAttention(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout=0.2):
        super().__init__()
        self.lstm      = nn.LSTM(input_size, hidden_size, num_layers,
                                 batch_first=True, bidirectional=True,
                                 dropout=dropout if num_layers > 1 else 0)
        # Attention with positional bias — later timesteps are more recent
        self.attention      = nn.Linear(hidden_size * 2, 1)
        self.pos_bias       = nn.Parameter(torch.zeros(1))  # learnable recency bias
        self.ln             = nn.LayerNorm(hidden_size * 2)
        self.dropout        = nn.Dropout(dropout)
        self.fc             = nn.Linear(hidden_size * 2, 2)
        
        # NEW: direct skip connection from last hidden to output
        self.skip_fc        = nn.Linear(hidden_size * 2, 2)
        self.output_blend   = nn.Parameter(torch.tensor(0.5))

    def forward(self, x):
        lstm_out, _  = self.lstm(x)                  # (B, T, H*2)
        seq_len      = lstm_out.size(1)
        
        # Recency-biased attention
        pos_weights  = torch.linspace(0, 1, seq_len, device=x.device).unsqueeze(-1)
        attn_raw     = self.attention(lstm_out) + self.pos_bias * pos_weights
        attn_weights = torch.softmax(attn_raw, dim=1)
        context      = (attn_weights * lstm_out).sum(dim=1)
        
        out_attn     = self.fc(self.dropout(self.ln(context)))
        
        # Skip: use last timestep directly (most recent information)
        last_hidden  = lstm_out[:, -1, :]
        out_skip     = self.skip_fc(last_hidden)
        
        # Blend — learnable, starts 50/50
        alpha        = torch.sigmoid(self.output_blend)
        return alpha * out_attn + (1 - alpha) * out_skip


# ── Joint Loss: Huber (magnitude) + Directional + Std-collapse penalty ────────
class JointPredictionLoss(nn.Module):
    def __init__(self, alpha=2.0, beta=0.5, gamma=1.0):
        super().__init__()
        self.alpha = alpha
        self.beta  = beta
        self.gamma = gamma  # NEW: magnitude-confidence penalty

    def forward(self, preds, targets, weights=None):
        # Huber for magnitude
        huber = nn.HuberLoss(delta=1.0, reduction="none")(preds, targets).mean(dim=1)

        # Directional loss — wrong sign with MAGNITUDE scaling
        # A prediction of -0.001 on +2.0 is penalized MORE than -0.001 on +0.1
        price_sign_loss  = torch.clamp(-preds[:, 0] * targets[:, 0], min=0) * targets[:, 0].abs()
        volume_sign_loss = torch.clamp(-preds[:, 1] * targets[:, 1], min=0) * targets[:, 1].abs()

        # Variance collapse penalty — directly penalize low prediction variance
        pred_std  = preds[:, 0].std() + 1e-8
        tgt_std   = targets[:, 0].std() + 1e-8
        std_ratio = pred_std / tgt_std
        # Penalize heavily when std_ratio < 0.3, ease off once > 0.7
        collapse_loss = torch.clamp(0.5 - std_ratio, min=0) ** 2

        # Bias penalty
        bias_penalty = preds[:, 0].mean() ** 2   # squared, not abs — smoother gradient

        per_sample = (
            huber
            + self.alpha * price_sign_loss
            + 0.3        * volume_sign_loss
        )
        if weights is not None:
            per_sample = per_sample * weights

        return (
            per_sample.mean()
            + self.beta  * std_ratio.reciprocal().clamp(max=10.0)  # NEW: reciprocal penalty
            + self.gamma * collapse_loss * 5.0
            + 2.0        * bias_penalty
        )
        
# ── Helpers ───────────────────────────────────────────────────────────────────
def create_sequences(X, y, seq_len, move_threshold=0.3):
    Xs, ys, ws = [], [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i : i + seq_len])
        ys.append(y[i + seq_len])
        move   = abs(y[i + seq_len, 0])
        ws.append(1.0 + 2.0 * float(move > move_threshold))
    return np.array(Xs), np.array(ys), np.array(ws, dtype=np.float32)


def make_balanced_sampler(y, oversample_factor=2.0):
    labels = (y[:, 0] > 0).astype(int)
    moves  = np.abs(y[:, 0])
    
    # Stratify into 4 buckets: UP-small, UP-large, DOWN-small, DOWN-large
    median_move = np.median(moves)
    bucket = labels * 2 + (moves > median_move).astype(int)
    # bucket: 0=DOWN-small, 1=DOWN-large, 2=UP-small, 3=UP-large
    
    counts = np.bincount(bucket, minlength=4).astype(float)
    counts = np.maximum(counts, 1)
    
    # Inverse frequency weights, with extra boost on large moves
    weight_map = 1.0 / counts
    weight_map[1] *= oversample_factor   # DOWN-large
    weight_map[3] *= oversample_factor   # UP-large
    
    sample_weights = torch.tensor(weight_map[bucket], dtype=torch.float32)
    return WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)


def safe_corr(a, b):
    """Pearson r, returns 0.0 on NaN / zero-std edge cases."""
    if np.std(a) < 1e-8 or np.std(b) < 1e-8:
        return 0.0
    r = np.corrcoef(a, b)[0, 1]
    return 0.0 if np.isnan(r) else float(r)


# ── Clean NaN values before Optuna ───────────────────────────────────────────
print("\nCleaning NaN values from all symbols...")
for symbol, data in prepared.items():
    for split in ("train", "val"):
        X_key, y_key = f"{split}_X_scaled", f"{split}_y_scaled"
        X, y         = data[X_key], data[y_key]
        mask         = ~(np.isnan(X).any(axis=1) | np.isnan(y).any(axis=1))
        before       = len(X)
        data[X_key]  = X[mask]
        data[y_key]  = y[mask]
        print(f"  {symbol} {split}: {before} → {len(data[X_key])} rows")


# ── Optuna search ─────────────────────────────────────────────────────────────
for symbol, data in prepared.items():
    print(f"\n{'═'*55}")
    print(f"  Optuna search — {symbol}")
    print(f"{'═'*55}")

    train_X    = data["train_X_scaled"]
    train_y    = data["train_y_scaled"]
    val_X      = data["val_X_scaled"]
    val_y      = data["val_y_scaled"]
    input_size = train_X.shape[1]

    def objective(trial):
        # ── Hyperparameters ───────────────────────────────────────────────────
        hidden_size = trial.suggest_categorical("hidden_size", [64, 128])
        num_layers  = trial.suggest_int("num_layers", 2, 3)
        dropout     = trial.suggest_float("dropout", 0.2, 0.4, step=0.1)
        lr          = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
        batch_size  = trial.suggest_categorical("batch_size", [32, 64])
        seq_len     = trial.suggest_categorical("seq_len", [10, 20, 30, 40])
        alpha       = trial.suggest_float("alpha", 3.0, 12.0, step=0.5)
        beta        = trial.suggest_float("beta",  0.3, 1.0, step=0.1)
        move_threshold = trial.suggest_float("move_threshold", 0.2, 0.5, step=0.1)

        # ── Sequences (now returns weights) ───────────────────────────────────
        X_tr, y_tr, w_tr = create_sequences(train_X, train_y, seq_len, move_threshold)
        X_vl, y_vl, _    = create_sequences(val_X,   val_y,   seq_len, move_threshold)

        sampler  = make_balanced_sampler(y_tr)
        train_ld = DataLoader(
            TadawulDataset(X_tr, y_tr, w_tr),
            batch_size=batch_size,
            sampler=sampler,
            drop_last=True
        )
        val_ld = DataLoader(
            TadawulDataset(X_vl, y_vl),          # no weights needed for val
            batch_size=batch_size,
            shuffle=False,
            drop_last=False
        )

        # ── Model & optimiser ─────────────────────────────────────────────────
        model = StockPctChangeBiLSTMAttention(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout
        ).to(device)

        criterion = JointPredictionLoss(alpha=alpha, beta=beta)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
        

        EPOCHS         = 30
        PATIENCE       = 7
        best_val_score = 0.0
        patience_ctr   = 0

        for epoch in range(1, EPOCHS + 1):
            # ── Train ─────────────────────────────────────────────────────────
            model.train()
            for X_batch, y_batch, w_batch in train_ld:   # ← unpack 3 items now
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                w_batch = w_batch.to(device)
                optimizer.zero_grad()
                loss = criterion(model(X_batch), y_batch, w_batch)  # ← pass weights
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            # ── Validate ──────────────────────────────────────────────────────
            model.eval()
            all_preds, all_actuals = [], []
            with torch.no_grad():
                for X_batch, y_batch, _ in val_ld:        # ← unpack 3 items
                    all_preds.append(model(X_batch.to(device)).cpu().numpy())
                    all_actuals.append(y_batch.numpy())

            if len(all_preds) == 0:
                return 0.0

            val_preds   = np.vstack(all_preds)
            val_actuals = np.vstack(all_actuals)

            # ── Direction (balanced accuracy) ─────────────────────────────────
            price_dir = balanced_accuracy_score(
                np.sign(val_actuals[:, 0]).astype(int),
                np.sign(val_preds[:, 0]).astype(int)
            )
            volume_dir = balanced_accuracy_score(
                np.sign(val_actuals[:, 1]).astype(int),
                np.sign(val_preds[:, 1]).astype(int)
            )

            # ── Magnitude correlation ─────────────────────────────────────────
            price_corr  = safe_corr(val_actuals[:, 0], val_preds[:, 0])
            volume_corr = safe_corr(val_actuals[:, 1], val_preds[:, 1])

            # ── Std-ratio penalty ─────────────────────────────────────────────
            pred_std_ratio = np.std(val_preds[:, 0]) / max(np.std(val_actuals[:, 0]), 1e-6)
            std_penalty    = abs(1.0 - pred_std_ratio) * 0.2

            # ── Collapse penalty ──────────────────────────────────────────────
            pred_up_pct    = (val_preds[:, 0] > 0).mean()
            pred_std_ratio = np.std(val_preds[:, 0]) / max(np.std(val_actuals[:, 0]), 1e-6)

            price_dir  = balanced_accuracy_score(
                np.sign(val_actuals[:, 0]).astype(int),
                np.sign(val_preds[:, 0]).astype(int)
            )
            price_corr = safe_corr(val_actuals[:, 0], val_preds[:, 0])

            val_score = (
                0.40 * price_dir
            + 0.35 * max(price_corr, 0.0)
            + 0.25 * min(pred_std_ratio, 1.0)   # reward variance up to actual level, not beyond
            - 0.5  * max(0.05 - pred_up_pct, 0) # soft penalty for near-zero UP predictions
            - 0.5  * max(pred_up_pct - 0.95, 0) # soft penalty for near-100% UP predictions
            )
            trial.report(val_score, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

            if val_score > best_val_score:
                best_val_score = val_score
                patience_ctr   = 0
            else:
                patience_ctr += 1
                if patience_ctr >= PATIENCE:
                    break

        return best_val_score

    # ── Run ───────────────────────────────────────────────────────────────────
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5),
        study_name=f"bilstm_attention_{symbol}"
    )
    study.optimize(objective, n_trials=50, timeout=3600)

    best_params_all[symbol] = study.best_params

    # ── Results ───────────────────────────────────────────────────────────────
    print(f"\n  Best joint score : {study.best_value:.4f}")
    print(f"  Best trial       : {study.best_trial.number}")
    print(f"  Best params:")
    for k, v in study.best_params.items():
        print(f"    {k:15s} : {v}")

    print(f"\n  Top 5 Trials:")
    trials_df = study.trials_dataframe().sort_values("value", ascending=False).head()
    cols = [
        "number", "value",
        "params_hidden_size", "params_num_layers", "params_dropout",
        "params_lr", "params_batch_size", "params_seq_len",
        "params_alpha", "params_beta", "params_move_threshold",
    ]
    print(trials_df[[c for c in cols if c in trials_df.columns]])

    print(f"\n── Optuna done for {len(best_params_all)} symbols ──")
    print(trials_df[[c for c in cols if c in trials_df.columns]])

    print(f"\n── Optuna done for {len(best_params_all)} symbols ──")

Using device: cuda

Cleaning NaN values from all symbols...
  7020 train: 3435 → 3435 rows
  7020 val: 736 → 736 rows
  2370 train: 2947 → 2947 rows
  2370 val: 632 → 632 rows

═══════════════════════════════════════════════════════
  Optuna search — 7020
═══════════════════════════════════════════════════════


[I 2026-05-16 04:05:47,735] A new study created in memory with name: bilstm_attention_7020
[I 2026-05-16 04:06:01,672] Trial 0 finished with value: 0.23251943914165274 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.00018410729205738696, 'batch_size': 32, 'seq_len': 10, 'alpha': 12.0, 'beta': 0.9000000000000001, 'move_threshold': 0.2}. Best is trial 0 with value: 0.23251943914165274.
[I 2026-05-16 04:06:12,922] Trial 1 finished with value: 0.3050294101629818 and parameters: {'hidden_size': 128, 'num_layers': 2, 'dropout': 0.30000000000000004, 'lr': 0.0005418282319533242, 'batch_size': 64, 'seq_len': 40, 'alpha': 10.0, 'beta': 0.4, 'move_threshold': 0.4}. Best is trial 1 with value: 0.3050294101629818.
[I 2026-05-16 04:06:20,890] Trial 2 finished with value: 0.2520507246113502 and parameters: {'hidden_size': 64, 'num_layers': 3, 'dropout': 0.2, 'lr': 0.00012897950480855554, 'batch_size': 64, 'seq_len': 10, 'alpha': 7.0, 'beta': 0.3, 'move_th


  Best joint score : 0.4801
  Best trial       : 28
  Best params:
    hidden_size     : 64
    num_layers      : 3
    dropout         : 0.4
    lr              : 0.0007908661148136831
    batch_size      : 32
    seq_len         : 10
    alpha           : 3.0
    beta            : 0.6000000000000001
    move_threshold  : 0.2

  Top 5 Trials:
    number     value  params_hidden_size  params_num_layers  params_dropout  \
28      28  0.480111                  64                  3             0.4   
12      12  0.470915                  64                  3             0.2   
27      27  0.460362                  64                  3             0.2   
19      19  0.459701                  64                  3             0.4   
11      11  0.458327                  64                  3             0.2   

    params_lr  params_batch_size  params_seq_len  params_alpha  params_beta  \
28   0.000791                 32              10           3.0          0.6   
12   0.004975       

[I 2026-05-16 04:17:06,610] Trial 0 finished with value: 0.5221708939077806 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.00018410729205738696, 'batch_size': 32, 'seq_len': 10, 'alpha': 12.0, 'beta': 0.9000000000000001, 'move_threshold': 0.2}. Best is trial 0 with value: 0.5221708939077806.
[I 2026-05-16 04:17:15,818] Trial 1 finished with value: 0.4835536350173397 and parameters: {'hidden_size': 128, 'num_layers': 2, 'dropout': 0.30000000000000004, 'lr': 0.0005418282319533242, 'batch_size': 64, 'seq_len': 40, 'alpha': 10.0, 'beta': 0.4, 'move_threshold': 0.4}. Best is trial 0 with value: 0.5221708939077806.
[I 2026-05-16 04:17:36,127] Trial 2 finished with value: 0.4940397579715297 and parameters: {'hidden_size': 64, 'num_layers': 3, 'dropout': 0.2, 'lr': 0.00012897950480855554, 'batch_size': 64, 'seq_len': 10, 'alpha': 7.0, 'beta': 0.3, 'move_threshold': 0.30000000000000004}. Best is trial 0 with value: 0.5221708939077806.
[I 2026-05-16


  Best joint score : 0.5300
  Best trial       : 27
  Best params:
    hidden_size     : 128
    num_layers      : 3
    dropout         : 0.4
    lr              : 0.0007227007114681773
    batch_size      : 32
    seq_len         : 10
    alpha           : 8.5
    beta            : 0.3
    move_threshold  : 0.4

  Top 5 Trials:
    number     value  params_hidden_size  params_num_layers  params_dropout  \
27      27  0.529997                 128                  3             0.4   
31      31  0.522238                 128                  3             0.4   
0        0  0.522171                 128                  3             0.3   
42      42  0.520829                 128                  3             0.3   
32      32  0.519559                 128                  3             0.4   

    params_lr  params_batch_size  params_seq_len  params_alpha  params_beta  \
27   0.000723                 32              10           8.5          0.3   
31   0.000518                 32  

In [7]:
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import balanced_accuracy_score
import numpy as np
import os
import time

trained_models = {}


# ── Validation metrics: direction + magnitude ─────────────────────────────────
def compute_val_metrics(model, loader, device):
    """
    Returns
    -------
    joint_score  : float  — same formula as Optuna objective
    price_dir    : float  — balanced accuracy, price direction
    volume_dir   : float  — balanced accuracy, volume direction
    price_corr   : float  — Pearson r, price magnitude
    volume_corr  : float  — Pearson r, volume magnitude
    pred_up_pct  : float  — fraction of UP predictions (collapse monitor)
    std_ratio    : float  — pred_std / actual_std  (1.0 = perfect)
    """
    model.eval()
    all_preds, all_actuals = [], []
    with torch.no_grad():
        for X_batch, y_batch, _ in loader:    # ← unpack 3
            all_preds.append(model(X_batch.to(device)).cpu().numpy())
            all_actuals.append(y_batch.numpy())

    preds   = np.vstack(all_preds)
    actuals = np.vstack(all_actuals)

    # ── Directional (balanced accuracy) ──────────────────────────────────────
    price_dir  = balanced_accuracy_score(
        np.sign(actuals[:, 0]).astype(int),
        np.sign(preds[:, 0]).astype(int)
    )
    volume_dir = balanced_accuracy_score(
        np.sign(actuals[:, 1]).astype(int),
        np.sign(preds[:, 1]).astype(int)
    )

    # ── Magnitude correlation (Pearson r) ─────────────────────────────────────
    def safe_corr(a, b):
        if np.std(a) < 1e-8 or np.std(b) < 1e-8:
            return 0.0
        r = np.corrcoef(a, b)[0, 1]
        return 0.0 if np.isnan(r) else float(r)

    price_corr  = safe_corr(actuals[:, 0], preds[:, 0])
    volume_corr = safe_corr(actuals[:, 1], preds[:, 1])

    # ── Std-ratio and collapse ────────────────────────────────────────────────
    pred_up_pct  = (preds[:, 0] > 0).mean()
    std_ratio    = np.std(preds[:, 0]) / max(np.std(actuals[:, 0]), 1e-6)
    std_penalty  = abs(1.0 - std_ratio) * 0.2
    collapse_penalty = 1.0 if (pred_up_pct < 0.05 or pred_up_pct > 0.95) else 0.0
    
    

    # ── Joint score (mirrors Optuna objective exactly) ────────────────────────
    joint_score = (
        0.35 * price_dir               +
        0.25 * volume_dir              +
        0.25 * max(price_corr,  0.0)   +
        0.15 * max(volume_corr, 0.0)   -
        std_penalty                    -
        collapse_penalty
    )

    return joint_score, price_dir, volume_dir, price_corr, volume_corr, pred_up_pct, std_ratio


# ── Training loop ─────────────────────────────────────────────────────────────
for symbol, data in prepared.items():
    print(f"\n{'═'*72}")
    print(f"  Training — {symbol}")
    print(f"{'═'*72}")

    best = best_params_all[symbol]
    print(f"  Best params: {best}")

    move_threshold = best.get("move_threshold", 0.3)   # fallback if not in best_params

    # ── Rebuild sequences with best seq_len ───────────────────────────────────
    X_train_f, y_train_f, w_train_f = create_sequences(data["train_X_scaled"], data["train_y_scaled"], best["seq_len"], move_threshold)
    X_val_f,   y_val_f,   _         = create_sequences(data["val_X_scaled"],   data["val_y_scaled"],   best["seq_len"], move_threshold)
    X_test_f,  y_test_f,  _         = create_sequences(data["test_X_scaled"],  data["test_y_scaled"],  best["seq_len"], move_threshold)

    # ── Loaders ───────────────────────────────────────────────────────────────
    sampler      = make_balanced_sampler(y_train_f)
    train_loader = DataLoader(
        TadawulDataset(X_train_f, y_train_f, w_train_f),   # ← pass weights
        batch_size=best["batch_size"],
        sampler=sampler,
        drop_last=True
    )
    val_loader  = DataLoader(TadawulDataset(X_val_f,  y_val_f),  batch_size=best["batch_size"], shuffle=False)
    test_loader = DataLoader(TadawulDataset(X_test_f, y_test_f), batch_size=best["batch_size"], shuffle=False)

    # ── Model ─────────────────────────────────────────────────────────────────
    model = StockPctChangeBiLSTMAttention(
        input_size  = data["train_X_scaled"].shape[1],
        hidden_size = best["hidden_size"],
        num_layers  = best["num_layers"],
        dropout     = best["dropout"]
    ).to(device)

    criterion = JointPredictionLoss(alpha=best["alpha"], beta=best["beta"])
    WARMUP_EPOCHS    = 10          # ramp alpha over first 10 epochs
    BASE_ALPHA       = best["alpha"]
    optimizer = torch.optim.Adam(model.parameters(), lr=best["lr"], weight_decay=1e-5)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

    # ── Config ────────────────────────────────────────────────────────────────
    EPOCHS           = 100
    PATIENCE         = 15
    best_joint_score = -np.inf
    patience_ctr     = 0
    history          = {
        "train_loss":  [],
        "val_loss":    [],
        "joint_score": [],
        "price_dir":   [],
        "price_corr":  [],
    }

    os.makedirs(f"models/{symbol}", exist_ok=True)
    save_path = f"models/{symbol}/{symbol}_best_bilstm.pt"

    # ── Header ────────────────────────────────────────────────────────────────
    print(f"\n{'Epoch':>6} | {'TrLoss':>8} | {'VaLoss':>8} | "
          f"{'Joint':>7} | {'PDir':>6} | {'VDir':>6} | "
          f"{'Pr':>6} | {'Vr':>6} | {'UP%':>5} | {'StdR':>5} | {'s':>5}")
    print("─" * 90)

    for epoch in range(1, EPOCHS + 1):
        start = time.time()
        
        warmup_scale = min(1.0, epoch / WARMUP_EPOCHS)
        criterion    = JointPredictionLoss(
            alpha = BASE_ALPHA * warmup_scale,
            beta  = best["beta"]
        )

        # ── Train ─────────────────────────────────────────────────────────────
        model.train()
        train_losses = []
        for X_batch, y_batch, w_batch in train_loader:       # ← unpack 3
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            w_batch = w_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch, w_batch)   # ← pass weights
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss.item())

        # ── Val loss ──────────────────────────────────────────────────────────
        model.eval()
        val_losses = []
        with torch.no_grad():
            for X_batch, y_batch, _ in val_loader:           # ← unpack 3
                val_losses.append(
                    criterion(model(X_batch.to(device)), y_batch.to(device)).item()
                )

        train_loss = np.mean(train_losses)
        val_loss   = np.mean(val_losses)

        # ── Val metrics ───────────────────────────────────────────────────────
        joint_score, p_dir, v_dir, p_corr, v_corr, pred_up_pct, std_ratio = \
            compute_val_metrics(model, val_loader, device)

        elapsed = time.time() - start

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["joint_score"].append(joint_score)
        history["price_dir"].append(p_dir)
        history["price_corr"].append(p_corr)

        print(f"{epoch:>6} | {train_loss:>8.5f} | {val_loss:>8.5f} | "
              f"{joint_score:>7.4f} | {p_dir:>6.3f} | {v_dir:>6.3f} | "
              f"{p_corr:>6.3f} | {v_corr:>6.3f} | {pred_up_pct:>4.0%} | "
              f"{std_ratio:>5.2f} | {elapsed:>4.1f}s")

        scheduler.step(val_loss)

        # ── Save on best joint score ───────────────────────────────────────────
        if joint_score > best_joint_score:
            best_joint_score = joint_score
            patience_ctr     = 0
            torch.save(model.state_dict(), save_path)
            print(f"         ✓ Saved  "
                  f"(p_dir={p_dir:.3f}, v_dir={v_dir:.3f}, "
                  f"p_corr={p_corr:.3f}, v_corr={v_corr:.3f}, "
                  f"up%={pred_up_pct:.0%}, std_r={std_ratio:.2f})")
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"\n  Early stopping at epoch {epoch}.")
                break

    trained_models[symbol] = {
        "best_joint_score": best_joint_score,
        "history":          history,
    }
    print(f"\n  Best joint score: {best_joint_score:.4f}")

print(f"\n── Training done for {len(trained_models)} symbols ──")
for s, r in trained_models.items():
    print(f"  {s}: best joint score = {r['best_joint_score']:.4f}")


════════════════════════════════════════════════════════════════════════
  Training — 7020
════════════════════════════════════════════════════════════════════════
  Best params: {'hidden_size': 64, 'num_layers': 3, 'dropout': 0.4, 'lr': 0.0007908661148136831, 'batch_size': 32, 'seq_len': 10, 'alpha': 3.0, 'beta': 0.6000000000000001, 'move_threshold': 0.2}

 Epoch |   TrLoss |   VaLoss |   Joint |   PDir |   VDir |     Pr |     Vr |   UP% |  StdR |     s
──────────────────────────────────────────────────────────────────────────────────────────
     1 |  3.07476 |  4.36161 |  0.1910 |  0.497 |  0.500 |  0.045 |  0.041 |  77% |  0.37 |  1.6s
         ✓ Saved  (p_dir=0.497, v_dir=0.500, p_corr=0.045, v_corr=0.041, up%=77%, std_r=0.37)
     2 |  3.04242 |  6.96647 |  0.1562 |  0.509 |  0.511 |  0.034 |  0.070 |  92% |  0.16 |  1.5s
     3 |  3.13977 |  6.86781 |  0.1580 |  0.508 |  0.500 |  0.029 |  0.089 |  88% |  0.17 |  1.7s
     4 |  3.26643 |  6.87667 |  0.1375 |  0.505 |  0.512 |  0

DIAGNOSTIC

In [11]:
# ── Evaluation on Test Set ────────────────────────────────────────────────────
print("\n" + "=" * 50)
print("EVALUATION ON TEST SET")
print("=" * 50)

def evaluate(actuals, preds, label):
    mae      = np.mean(np.abs(actuals - preds))
    rmse     = np.sqrt(np.mean((actuals - preds) ** 2))
    dir_acc  = np.mean(np.sign(actuals) == np.sign(preds)) * 100

    corr       = np.corrcoef(actuals, preds)[0, 1]
    pred_std   = np.std(preds)
    actual_std = np.std(actuals)
    within_1pct = np.mean(np.abs(actuals - preds) < 1.0) * 100
    within_2pct = np.mean(np.abs(actuals - preds) < 2.0) * 100

    print(f"\n  [{label}]")
    print(f"  MAE             : {mae:.4f}%")
    print(f"  RMSE            : {rmse:.4f}%")
    print(f"  Pearson r       : {corr:.4f}")
    print(f"  Pred std        : {pred_std:.4f}  Actual std: {actual_std:.4f}")
    print(f"  Within ±1%      : {within_1pct:.1f}%")
    print(f"  Within ±2%      : {within_2pct:.1f}%")
    print(f"  Directional Acc : {dir_acc:.2f}%")

for symbol in symbols:
    if symbol not in prepared:
        print(f"\n  Skipping {symbol} — not in prepared dict.")
        continue
    if symbol not in trained_models:
        print(f"\n  Skipping {symbol} — no trained model found.")
        continue

    print(f"\n{'═'*55}")
    print(f"  Evaluating — {symbol}")
    print(f"{'═'*55}")

    best = best_params_all[symbol]
    data = prepared[symbol]
    move_threshold = best.get("move_threshold", 0.3)

    # ── Rebuild test loader with best seq_len ─────────────────────────────────
    X_test_f, y_test_f, _ = create_sequences(          # ← unpack 3, discard w
        data["test_X_scaled"], data["test_y_scaled"], best["seq_len"], move_threshold
    )
    test_loader = DataLoader(
        TadawulDataset(X_test_f, y_test_f),            # no weights needed for eval
        batch_size=best["batch_size"],
        shuffle=False
    )

    # ── Load best model ───────────────────────────────────────────────────────
    checkpoint = torch.load(
        f"models/{symbol}/{symbol}_best_bilstm.pt",
        weights_only=True
    )

    inferred_hidden_size = checkpoint["lstm.weight_ih_l0"].shape[0] // 4
    inferred_num_layers  = sum(
        1 for k in checkpoint if k.startswith("lstm.weight_ih_l") and "_reverse" not in k
    )

    model = StockPctChangeBiLSTMAttention(
        input_size  = data["train_X_scaled"].shape[1],
        hidden_size = inferred_hidden_size,
        num_layers  = inferred_num_layers,
        dropout     = best["dropout"]
    ).to(device)

    model.load_state_dict(checkpoint)
    model.eval()

    # ── Load target scaler ────────────────────────────────────────────────────
    with open(f"models/{symbol}/{symbol}_target_scaler.pkl", "rb") as f:
        target_scaler = pickle.load(f)

    # ── Inference ─────────────────────────────────────────────────────────────
    all_preds, all_actuals = [], []
    with torch.no_grad():
        for X_batch, y_batch, _ in test_loader:        # ← unpack 3
            preds   = model(X_batch.to(device)).cpu().numpy()
            actuals = y_batch.numpy()
            all_preds.append(preds)
            all_actuals.append(actuals)

    all_preds   = np.vstack(all_preds)
    all_actuals = np.vstack(all_actuals)

    # ── Inverse transform ─────────────────────────────────────────────────────
    all_preds   = target_scaler.inverse_transform(all_preds)
    all_actuals = target_scaler.inverse_transform(all_actuals)

    price_preds_pct    = all_preds[:, 0]
    price_actuals_pct  = all_actuals[:, 0]
    volume_preds_pct   = np.expm1(all_preds[:, 1])   * 100
    volume_actuals_pct = np.expm1(all_actuals[:, 1]) * 100

    # ── Metrics ───────────────────────────────────────────────────────────────
    evaluate(price_actuals_pct,  price_preds_pct,  "Price % Change")
    evaluate(volume_actuals_pct, volume_preds_pct, "Volume % Change")

    # ── Save predictions ──────────────────────────────────────────────────────
    results_df = pd.DataFrame({
        "actual_price_pct"    : price_actuals_pct,
        "predicted_price_pct" : price_preds_pct,
        "actual_volume_pct"   : volume_actuals_pct,
        "predicted_volume_pct": volume_preds_pct,
    })
    out_path = f"models/{symbol}/{symbol}_test_predictions.csv"
    results_df.to_csv(out_path, index=False)
    print(f"\n  Predictions saved: {out_path}")

print(f"\n── Evaluation done for {len(symbols)} symbols ──")


EVALUATION ON TEST SET

═══════════════════════════════════════════════════════
  Evaluating — 7020
═══════════════════════════════════════════════════════

  [Price % Change]
  MAE             : 2.1854%
  RMSE            : 2.5963%
  Pearson r       : 0.0712
  Pred std        : 0.8541  Actual std: 1.5051
  Within ±1%      : 23.4%
  Within ±2%      : 46.9%
  Directional Acc : 47.87%

  [Volume % Change]
  MAE             : 54.0740%
  RMSE            : 72.1533%
  Pearson r       : 0.0081
  Pred std        : 1.7284  Actual std: 71.1162
  Within ±1%      : 1.4%
  Within ±2%      : 2.3%
  Directional Acc : 51.31%

  Predictions saved: models/7020/7020_test_predictions.csv

═══════════════════════════════════════════════════════
  Evaluating — 2370
═══════════════════════════════════════════════════════

  [Price % Change]
  MAE             : 2.6983%
  RMSE            : 3.3898%
  Pearson r       : 0.0187
  Pred std        : 1.8835  Actual std: 2.2184
  Within ±1%      : 23.6%
  Within ±2%  

In [12]:
import pandas as pd
import numpy as np

for symbol in symbols:
    csv_path = f"models/{symbol}/{symbol}_test_predictions.csv"

    try:
        results = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"\n  Skipping {symbol} — {csv_path} not found.")
        continue

    price_actual    = results["actual_price_pct"]
    price_predicted = results["predicted_price_pct"]

    correct   = (np.sign(price_actual) == np.sign(price_predicted))
    up_mask   = price_actual > 0
    down_mask = price_actual < 0

    print(f"\n{'═'*55}")
    print(f"  {symbol} — Honest Model Baseline")
    print(f"{'═'*55}")
    print(f"  Actual    mean : {price_actual.mean():.4f}   std: {price_actual.std():.4f}")
    print(f"  Predicted mean : {price_predicted.mean():.4f}   std: {price_predicted.std():.4f}")
    print(f"\n  Overall Dir Acc : {correct.mean()*100:.2f}%")
    print(f"  When actual UP  : {correct[up_mask].mean()*100:.2f}%  ({up_mask.sum()} samples)")
    print(f"  When actual DOWN: {correct[down_mask].mean()*100:.2f}%  ({down_mask.sum()} samples)")
    print(f"\n  % predicted UP  : {(price_predicted > 0).mean()*100:.2f}%")
    print(f"  % actual UP     : {(price_actual > 0).mean()*100:.2f}%")

    print(f"\n  Dir Acc by Move Size:")
    bins   = [0, 0.5, 1.0, 2.0, 5.0, 100.0]
    labels = ["<0.5%", "0.5-1%", "1-2%", "2-5%", ">5%"]
    price_actual_abs = price_actual.abs()
    for i, label in enumerate(labels):
        mask = (price_actual_abs >= bins[i]) & (price_actual_abs < bins[i+1])
        if mask.sum() > 0:
            acc = correct[mask].mean() * 100
            print(f"    {label:>8} moves : {acc:.2f}%  ({mask.sum()} samples)")

print(f"\n── Diagnostics done for {len(symbols)} symbols ──")


═══════════════════════════════════════════════════════
  7020 — Honest Model Baseline
═══════════════════════════════════════════════════════
  Actual    mean : 0.0470   std: 1.5062
  Predicted mean : -1.9352   std: 0.8547

  Overall Dir Acc : 47.87%
  When actual UP  : 0.00%  (379 samples)
  When actual DOWN: 100.00%  (348 samples)

  % predicted UP  : 0.00%
  % actual UP     : 52.13%

  Dir Acc by Move Size:
       <0.5% moves : 41.36%  (220 samples)
      0.5-1% moves : 54.82%  (166 samples)
        1-2% moves : 51.15%  (217 samples)
        2-5% moves : 45.00%  (120 samples)
         >5% moves : 25.00%  (4 samples)

═══════════════════════════════════════════════════════
  2370 — Honest Model Baseline
═══════════════════════════════════════════════════════
  Actual    mean : -0.0018   std: 2.2202
  Predicted mean : -1.7845   std: 1.8850

  Overall Dir Acc : 51.29%
  When actual UP  : 20.13%  (298 samples)
  When actual DOWN: 79.94%  (324 samples)

  % predicted UP  : 20.10%
  % a

In [10]:
for symbol in symbols:
    df = pd.read_csv(f'{symbol}_daily.csv', parse_dates=["datetime"])
    n = len(df)
    val_df  = df.iloc[int(n*0.70):int(n*0.85)]
    test_df = df.iloc[int(n*0.85):]
    
    print(f"\n{symbol}")
    print(f"  Val  UP%: {(val_df['price_pct_change'] > 0).mean():.2%}  ({val_df['datetime'].min().date()} → {val_df['datetime'].max().date()})")
    print(f"  Test UP%: {(test_df['price_pct_change'] > 0).mean():.2%}  ({test_df['datetime'].min().date()} → {test_df['datetime'].max().date()})")


7020
  Val  UP%: 47.34%  (2020-01-19 → 2023-03-13)
  Test UP%: 48.86%  (2023-03-14 → 2026-05-13)

2370
  Val  UP%: 44.51%  (2020-11-19 → 2023-08-20)
  Test UP%: 49.12%  (2023-08-21 → 2026-05-13)
